In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re

BASE_URL   = "https://baniola.tn/voitures"
OUTPUT_CSV = "baniola_cars_full.csv"
DELAY_MIN  = 1.5
DELAY_MAX  = 3.0
MAX_PAGES  = None  # Set to None to scrape all pages

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Final columns to keep (in order)
KEEP_COLS = ["url", "title", "price_raw", "price_dt", "year", "km", "km_num",
             "fuel", "gearbox", "region", "posted", "negotiable", "estimation", "exchange", "image"]

In [2]:
def parse_card(card):
    result = {}

    # --- URL ---
    link = card.select_one("a.stretched-link")
    result["url"] = link["href"] if link else ""

    # --- Image ---
    img = card.select_one("img.ad-img")
    result["image"] = img.get("src", "") if img else ""

    # --- Price ---
    price_el = card.select_one(".card-price")
    if price_el:
        raw = price_el.get_text(" ", strip=True)
        result["price_raw"] = re.sub(r"\s+", " ", raw).strip()
        nums = re.sub(r"[^\d]", "", result["price_raw"])
        result["price_dt"] = int(nums) if nums else None
    else:
        result["price_raw"] = "Prix sur demande"
        result["price_dt"] = None

    # --- Title ---
    title_el = card.select_one("h3.ad-title")
    result["title"] = title_el.get_text(strip=True) if title_el else ""

    # --- Specs: year, km, fuel ---
    # Real class is .spec-pill, not .spec-item
    result.update({"year": "", "km": "", "km_num": None, "fuel": "", "gearbox": ""})
    for pill in card.select(".spec-pill"):
        txt = pill.get_text(strip=True)
        if re.match(r"^\d{4}$", txt):
            result["year"] = txt
        elif re.search(r"\d.*km", txt, re.IGNORECASE):
            result["km"] = txt
            result["km_num"] = int(re.sub(r"[^\d]", "", txt)) or None
        elif txt in ("Essence", "Diesel", "Hybride", "Électrique", "GPL"):
            result["fuel"] = txt
        elif txt in ("Manuelle", "Automatique"):
            result["gearbox"] = txt

    # --- Region ---
    region_el = card.select_one(".card-region")
    result["region"] = region_el.get_text(strip=True) if region_el else ""

    # --- Posted time ---
    date_el = card.select_one(".card-date")
    result["posted"] = date_el.get_text(strip=True) if date_el else ""

    # --- Negotiable badge ---
    neg = card.select_one(".badge-trust")
    result["negotiable"] = neg.get_text(strip=True) if neg else ""

    # --- Market estimation badge ---
    est = card.select_one(".estimation-badge")
    result["estimation"] = est.get_text(strip=True) if est else ""

    # --- Exchange badge ---
    exch = card.select_one(".btn-exchange")
    result["exchange"] = exch.get_text(strip=True) if exch else ""

    return result


def scrape_page(page_num):
    url = f"{BASE_URL}?page={page_num}"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  ⚠️  Page {page_num} failed: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.select("article.annonce-card")

    if not cards:
        print(f"  ⚠️  No cards on page {page_num}")
        return []

    results = []
    for card in cards:
        try:
            data = parse_card(card)
            results.append(data)
        except Exception as e:
            print(f"  ⚠️  Error parsing card: {e}")
    return results

In [3]:
def get_last_page():
    resp = requests.get(f"{BASE_URL}?page=1", headers=HEADERS, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")

    page_nums = []
    for a in soup.select("a[href*='page=']"):
        m = re.search(r"page=(\d+)", a.get("href", ""))
        if m:
            page_nums.append(int(m.group(1)))
    # Also check text-based pagination buttons
    for a in soup.select("a.page-link, [class*='pagination'] a"):
        txt = a.get_text(strip=True)
        if txt.isdigit():
            page_nums.append(int(txt))

    last = max(page_nums) if page_nums else 1
    print(f"✅ Last page detected: {last}")
    return last

In [4]:
last_page = get_last_page()
pages_to_scrape = min(MAX_PAGES, last_page) if MAX_PAGES else last_page

all_cars = []
for page in range(1, pages_to_scrape + 1):
    print(f"Scraping page {page}/{pages_to_scrape}...", end=" ")
    cars = scrape_page(page)
    print(f"→ {len(cars)} listings")
    all_cars.extend(cars)
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

print(f"\n✅ Total: {len(all_cars)} listings scraped")

✅ Last page detected: 70
Scraping page 1/70... → 50 listings
Scraping page 2/70... → 50 listings
Scraping page 3/70... → 50 listings
Scraping page 4/70... → 50 listings
Scraping page 5/70... → 50 listings
Scraping page 6/70... → 50 listings
Scraping page 7/70... → 50 listings
Scraping page 8/70... → 50 listings
Scraping page 9/70... → 50 listings
Scraping page 10/70... → 50 listings
Scraping page 11/70... → 50 listings
Scraping page 12/70... → 50 listings
Scraping page 13/70... → 50 listings
Scraping page 14/70... → 50 listings
Scraping page 15/70... → 50 listings
Scraping page 16/70... → 50 listings
Scraping page 17/70... → 50 listings
Scraping page 18/70... → 50 listings
Scraping page 19/70... → 50 listings
Scraping page 20/70... → 50 listings
Scraping page 21/70... → 50 listings
Scraping page 22/70... → 50 listings
Scraping page 23/70... → 50 listings
Scraping page 24/70... → 50 listings
Scraping page 25/70... → 50 listings
Scraping page 26/70... → 50 listings
Scraping page 27/70...

In [5]:
df = pd.DataFrame(all_cars)

# Keep only the columns we want, in the right order
df = df[[c for c in KEEP_COLS if c in df.columns]]

print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nNon-null counts:\n{df.notna().sum()}")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Saved {len(df)} rows to {OUTPUT_CSV}")

                                                 url  \
0  https://baniola.tn/details/30554/toyota-coroll...   
1  https://baniola.tn/details/30400/volkswagen-ti...   
2  https://baniola.tn/details/30583/fiat-500-2010...   
3  https://baniola.tn/details/30538/volkswagen-go...   
4  https://baniola.tn/details/30607/dacia-dokker-...   

                               title              price_raw  price_dt  year  \
0        Toyota Corolla 2026 Essence  105 000 DT Négociable  105000.0  2026   
1     Volkswagen Tiguan 2018 Essence   74 000 DT Négociable   74000.0  2018   
2              Fiat 500 2010 Essence   26 000 DT Négociable   26000.0  2010   
3  Volkswagen Golf Plus 2019 Essence   78 000 DT Négociable   78000.0  2019   
4           Dacia Dokker 2020 Diesel              30 000 DT   30000.0  2020   

           km    km_num     fuel gearbox   region           posted negotiable  \
0      800 km     800.0  Essence          Tunisie  Il y a 8 heures              
1  105 000 km  105000.0  E